In [93]:
import awswrangler as wr

In [ ]:
def exec_sql(sql_statement:str, db="financials"):
  return wr.athena.read_sql_query(
        sql=sql_statement,
        database=db
    )

In [95]:
sql ="""
    SELECT *
    FROM silver_t212_positions
    ORDER BY ingested_date DESC
    limit 10;
"""

df_positions = exec_sql(sql)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


In [ ]:
sql ="""
    WITH account_summary AS (
      SELECT 
          ROW_NUMBER() OVER (PARTITION BY ingested_date ORDER BY ingested_timestamp DESC) as rn
        , total_value_investmented
      FROM silver_t212_account_summary
    limit 10
    )
    SELECT *
    FROM cte
    WHERE rn = 1
    ORDER BY ingested_date DESC
"""

df_account_summary = exec_sql(sql)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


In [97]:
df_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   name                            10 non-null     string        
 1   ticker                          10 non-null     string        
 2   isin                            10 non-null     string        
 3   created_at                      10 non-null     datetime64[ns]
 4   asset_currency                  10 non-null     string        
 5   avg_price_paid                  10 non-null     float64       
 6   current_price                   10 non-null     float64       
 7   quantity                        10 non-null     float64       
 8   quantity_available_for_trading  10 non-null     float64       
 9   quantity_in_pies                10 non-null     float64       
 10  account_currency                10 non-null     string        
 11  current_value       

In [98]:
df_account_summary.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   id                        6 non-null      Int64         
 1   currency                  6 non-null      string        
 2   total_value               6 non-null      float64       
 3   cash_available_to_trade   6 non-null      float64       
 4   cash_reserved_for_orders  6 non-null      Int64         
 5   cash_in_pies              6 non-null      float64       
 6   total_value_investmented  6 non-null      float64       
 7   total_investment_cost     6 non-null      float64       
 8   realized_profit_loss      6 non-null      float64       
 9   unrealized_profit_loss    6 non-null      float64       
 10  ingested_timestamp        6 non-null      datetime64[ns]
 11  ingested_date             6 non-null      object        
dtypes: Int64(2), datetime64[ns](1), float

In [ ]:
from datetime import datetime

In [210]:
from_date = "2026-08-17"
to_date = "2026-09-25"

sql =F"""

WITH account_summary AS (
    SELECT
        ROW_NUMBER() OVER (
            PARTITION BY ingested_date
            ORDER BY ingested_timestamp DESC
        ) AS rn,
        ingested_date,
        total_investment_cost,
        unrealized_profit_loss AS account_pnl
    FROM silver_t212_account_summary
    WHERE (
          ingested_date >= date_parse('{from_date}', '%Y-%m-%d') - interval '364' day
      AND ingested_date <= date_parse('{to_date}', '%Y-%m-%d')
      )
),

positions AS (
    SELECT
        ROW_NUMBER() OVER (
            PARTITION BY ingested_date, ticker
            ORDER BY ingested_timestamp DESC
        ) AS rn,
        name,
        ticker,
        ingested_date,
        ingested_timestamp,
        quantity,
        current_price,
        current_value,
        unrealized_profit_loss AS pnl
    FROM silver_t212_positions
    WHERE ticker='VWRPl_EQ'
    AND (
          ingested_date >= date_parse('{from_date}', '%Y-%m-%d') - interval '364' day
      AND ingested_date <= date_parse('{to_date}', '%Y-%m-%d')
      )
),

daily_positions AS (
    SELECT
        name,
        ticker,
        ingested_date,
        quantity,
        current_price,
        current_value,
        pnl,

        LAG(current_price) OVER (
            PARTITION BY ticker
            ORDER BY ingested_date
        ) AS prev_price,

        LAG(current_value) OVER (
            PARTITION BY ticker
            ORDER BY ingested_date
        ) AS prev_value,

        AVG(current_value) OVER (
            PARTITION BY ticker
            ORDER BY ingested_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS value_7d_ma,

        AVG(current_value) OVER (
            PARTITION BY ticker
            ORDER BY ingested_date
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) AS value_30d_ma,

        AVG(current_value) OVER (
            PARTITION BY ticker
            ORDER BY ingested_date
            ROWS BETWEEN 89 PRECEDING AND CURRENT ROW
        ) AS value_90d_ma,

        AVG(current_value) OVER (
            PARTITION BY ticker
            ORDER BY ingested_date
            ROWS BETWEEN 179 PRECEDING AND CURRENT ROW
        ) AS value_180d_ma,

        AVG(current_value) OVER (
            PARTITION BY ticker
            ORDER BY ingested_date
            ROWS BETWEEN 364 PRECEDING AND CURRENT ROW
        ) AS value_365d_ma

    FROM positions
    WHERE rn = 1
)

SELECT
    p.name,
    p.ticker,
    p.ingested_date AS data_date,
    p.quantity,
    p.current_price,
    p.prev_price,
    p.current_price - p.prev_price AS daily_price_change,
    (
        p.current_price - p.prev_price
    ) / NULLIF(p.prev_price, 0) * 100 AS daily_price_change_pct,
    p.current_value,
    p.prev_value,
    p.current_value - p.prev_value AS daily_change,
    (
        p.current_value - p.prev_value
    ) / NULLIF(p.prev_value, 0) * 100 AS daily_change_pct,
    p.pnl,
    p.pnl / NULLIF(a.account_pnl, 0) * 100 AS pct_account_pnl,
    p.current_value
        / NULLIF(a.total_investment_cost, 0) * 100 AS pct_weight,
    value_7d_ma,
    value_30d_ma,
    value_90d_ma,
    value_180d_ma,
    value_365d_ma
FROM daily_positions p
INNER JOIN account_summary a
    ON p.ingested_date = a.ingested_date
    AND a.rn = 1
WHERE prev_value IS NOT NULL
  AND p.ingested_date >= date_parse('{from_date}', '%Y-%m-%d')
ORDER BY data_date ;
"""

df_fact = exec_sql(sql)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


In [ ]:
df_fact.shape[1]
df_fact.shape[0]

40

In [202]:
df_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   name                    38 non-null     string 
 1   ticker                  38 non-null     string 
 2   data_date               38 non-null     object 
 3   quantity                38 non-null     float64
 4   current_price           38 non-null     float64
 5   prev_price              38 non-null     float64
 6   daily_price_change      38 non-null     float64
 7   daily_price_change_pct  38 non-null     float64
 8   current_value           38 non-null     float64
 9   prev_value              38 non-null     float64
 10  daily_change            38 non-null     float64
 11  daily_change_pct        38 non-null     float64
 12  pnl                     38 non-null     float64
 13  pct_return              38 non-null     float64
 14  pct_weight              38 non-null     float64
 15  ma

In [198]:
df_fact.groupby(["ticker"]).data_date.min()

ticker
VWRPl_EQ    2026-08-17
Name: data_date, dtype: object

In [211]:
df_fact[df_fact["daily_price_change"] <= -1].sort_values("data_date")

,name,ticker,data_date,quantity,current_price,prev_price,daily_price_change,daily_price_change_pct,current_value,prev_value,daily_change,daily_change_pct,pnl,pct_account_pnl,pct_weight,value_7d_ma,value_30d_ma,value_90d_ma,value_180d_ma,value_365d_ma
1,Vanguard FTSE All-World (Acc),VWRPl_EQ,2026-08-18,7.548378,142.803,144.557,-1.754,-1.213362,1260.37,1276.82,-16.45,-1.288357,69.65,6.608097,14.833065,1157.461429,1103.264667,1101.402581,1101.402581,1101.402581
15,Vanguard FTSE All-World (Acc),VWRPl_EQ,2026-09-01,7.548378,142.718,144.220,-1.502,-1.041464,1257.00,1269.77,-12.77,-1.005694,66.28,6.628729,14.806595,1265.895714,1189.691333,1150.753556,1150.753556,1150.753556
23,Vanguard FTSE All-World (Acc),VWRPl_EQ,2026-09-09,7.548378,142.543,143.858,-1.315,-0.914096,1253.53,1264.84,-11.31,-0.894184,62.81,6.025171,14.756022,1264.755714,1232.184667,1167.859245,1167.859245,1167.859245
28,Vanguard FTSE All-World (Acc),VWRPl_EQ,2026-09-14,7.548378,142.157,143.220,-1.063,-0.742215,1253.25,1259.75,-6.50,-0.515975,62.53,6.269677,14.749809,1256.884286,1256.053000,1175.454483,1175.454483,1175.454483


In [ ]:
"""
t212_asset_dim

name:
ticker:


"""


In [ ]:
from_date = "2026-08-17"
to_date = "2026-09-25"

sql =F"""
WITH asset_mapping AS (
  SELECT *
  FROM
  asset_mapping
  
)
MERGE INTO t212_asset_dim t
  USING silver_t212_positions s
  ON t.ticker = s.ticker
    AND t.name = s.name
    
  
WHEN MATCHED THEN
  UPDATE SET
  
"""

t212_asset_dim = exec_sql(sql)